# 路径规划一（TSP, Travelling Saleman Problem）

## 需求

> 给出N个位置点：0,1,2...,N-1。要求从0点出发，访问所有节点一次回到0点的最短路径

### 方案一

> 最容易想到的是暴力破解，遍历所有的情况，时间复杂度是O(n!)，适合n非常小的场景

### 方案二

> 使用贪心算法，每次找距离当前位置最近的下一个点，时间复杂度是O(n^2), 结果不是最优解，适合对时间敏感但结果要求不高的场景

### 方案三

> 使用动态规划，时间、空间复杂度均为O(n^2·2^n)，适合n＜20的场景
> 1. 算出两两之间的距离dist, dist[i][j]表示i到j的距离
> 2. dp[mask][i]表示已经走过mask表示的节点，最后停留在i上的最短距离，mask是二进制掩码，例如1001，表示已经走过节点0和3，dp[1001][3]表示从节点0出发到节点3，当前停留在节点三上
> 3. 状态转移方程：dp[mask][j] = min(dp[mask^(1 << j)][i] + dist[i][j]), i≠j 且 i∈mask

In [3]:
def tsp_dp(dist):
    n = len(dist)
    INF = float('inf')
    FULL = (1 << n) - 1
    
    dp = [[INF] * n for _ in range(1 << n)]
    dp[1 << 0][0] = 0
    path = [[-1] * n for _ in range(1 << n)]

    for mask in range(1, 1 << n):
        if not (mask & 1): # start from node0
            continue
        
        for i in range(n):
            if not (mask & (1 <<i)): # i must in mask
                continue
            if dp[mask][i] == INF:
                continue
            for j in range(n):
                if mask & (1 << j):
                    continue
                new_mask = mask | (1 << j)
                new_cost = dp[mask][i] + dist[i][j]
                if new_cost < dp[new_mask][j]:
                    dp[new_mask][j] = new_cost
                    path[new_mask][j] = i
    
    best_cost = INF
    best_last = -1

    for k in range(1, n):
        if dp[FULL][k] == INF:
            continue
        cost = dp[FULL][k] + dist[k][0]
        if cost < best_cost:
            best_cost = cost
            best_last = k
    
    best_path = []
    cur = best_last
    mask = FULL
    while cur != -1:
        best_path.append(cur)
        prev = path[mask][cur]
        mask ^= (1 << cur)
        cur = prev
    best_path.reverse()
    best_path.append(0)

    return best_cost, best_path

        
dist = [
    [0, 10, 20, 15],
    [10,  0, 12, 18],
    [20, 12,  0,  8],
    [15, 18,  8,  0]
]

cost, path = tsp_dp(dist)
city = ['A', 'B', 'C', 'D']
print(f"最短路程: {cost}")
print(f"最优路径: {' → '.join(city[i] for i in path)}")

最短路程: 45
最优路径: A → D → C → B → A
